# Day 8: Lyapunov Stability — Building a Crisis Detector from First Principles

*Alpha Flow Research · HongJin HE · July 2026*

---

## The Question: Is This Market Stable?

The biggest challenge in quantitative finance is **regime detection**: identifying when market dynamics have fundamentally shifted.
Standard approaches (VIX threshold, rolling vol, drawdown rules) are all *lagging indicators* — they react after the crisis has begun.

We take a dynamical systems approach. A market is in a **stable regime** when there exists a Lyapunov function $V: \mathcal{Z} \to \mathbb{R}_{\ge 0}$ satisfying the **stochastic Lyapunov condition**:

$$\mathcal{L}V(z) \le -c \cdot V(z) + b$$

where $\mathcal{L}$ is the generator of the latent state process $z_t$.

When this fails — when $\mathcal{L}V(z) > 0$ — the system is **drifting away from equilibrium**.

## The Crisis Indicator

We construct a real-time crisis indicator:

$$\Lambda_t = \frac{\mathcal{L}V(z_t)}{V(z_t)}$$

- $\Lambda_t < 0$: stable regime
- $\Lambda_t > 0$: unstable — system drifting away from invariant measure
- $\Lambda_t \gg 0$: crisis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import uniform_filter1d

np.random.seed(2026)

# ── Simulate a latent state with regime switches ──────────────────────────
# z_t = (vol_t, momentum_t, correlation_t)
T = 252 * 10  # 10 years

# Define crisis periods (inspired by real market events)
crises = [
    (252*2, 252*2+60, 'GFC 2008'),    # year 2-3
    (252*5, 252*5+20, 'Flash Crash'), # year 5  
    (252*7, 252*7+45, 'COVID 2020'),  # year 7
]

# Stable regime dynamics: mean-reverting OU process
theta, mu_eq, sigma_stable = 0.1, 0.2, 0.03  # OU parameters
z = np.zeros(T)
z[0] = mu_eq

for t in range(1, T):
    in_crisis = any(t_start <= t <= t_end for t_start, t_end, _ in crises)
    if in_crisis:
        # Crisis: drift flips direction, vol spikes
        drift = +0.3 * (z[t-1] - mu_eq)  # positive feedback (unstable)
        noise = 0.15 * np.random.randn()
    else:
        # Stable: mean-reverting
        drift = -theta * (z[t-1] - mu_eq)
        noise = sigma_stable * np.random.randn()
    z[t] = z[t-1] + drift + noise
    z[t] = np.clip(z[t], -2, 4)  # physical bounds

# ── Compute Lyapunov function V(z) = z² (quadratic, simplest case) ──────
V = z**2

# Approximate generator: LV ≈ ΔV/Δt (finite differences)
dV = np.diff(V)
LV = dV  # in discrete time, LV(z_t) ≈ E[V(z_{t+1}) - V(z_t) | z_t]
# Smooth with rolling window
LV_smooth = uniform_filter1d(LV, size=21)
V_smooth = V[:-1]

# Crisis indicator
Lambda = LV_smooth / np.maximum(V_smooth, 0.01)

t_arr = np.arange(T-1)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Shade crisis periods
for ax in axes:
    for t_start, t_end, label in crises:
        ax.axvspan(t_start, t_end, alpha=0.2, color='red')

axes[0].plot(t_arr, z[:-1], color='steelblue', lw=0.8, label='z_t (latent vol state)')
axes[0].axhline(mu_eq, ls='--', color='black', alpha=0.5, label=f'Equilibrium μ={mu_eq}')
axes[0].set_ylabel('Latent state z_t')
axes[0].legend()
axes[0].set_title('Lyapunov-Based Crisis Detection (Red = Known Crisis Periods)')

axes[1].plot(t_arr, V_smooth, color='orange', lw=0.8, label='V(z_t) = z²')
axes[1].set_ylabel('V(z_t)')
axes[1].legend()

axes[2].plot(t_arr, Lambda, color='crimson', lw=0.8, alpha=0.8, label='Λ_t = LV(z)/V(z)')
axes[2].axhline(0, color='black', lw=1.5, ls='--')
axes[2].fill_between(t_arr, Lambda, 0, where=Lambda > 0, alpha=0.4, color='red', label='Unstable (Λ>0)')
axes[2].fill_between(t_arr, Lambda, 0, where=Lambda < 0, alpha=0.2, color='green', label='Stable (Λ<0)')
axes[2].set_ylabel('Crisis indicator Λ_t')
axes[2].set_xlabel('Trading day')
axes[2].legend()
axes[2].set_ylim(-5, 8)

# Add crisis labels
for t_start, t_end, label in crises:
    axes[0].text((t_start+t_end)/2, max(z)*0.9, label, ha='center', fontsize=8, color='red')

plt.tight_layout()
plt.savefig('../figures/lyapunov_crisis_detector.png', dpi=150, bbox_inches='tight')
plt.show()

## Theoretical Foundation

**Theorem (Stochastic Lyapunov)**: Let $V \in C^2(\mathcal{Z})$, $V \ge 0$. If there exist constants $c > 0, b \ge 0$ such that:
$$\mathcal{L}V(z) = \nabla V(z) \cdot f(z) + \frac{1}{2} \text{tr}(\sigma \sigma^\top \nabla^2 V(z)) \le -cV(z) + b$$
then the SDE has a unique invariant probability measure $\mu^* \in \mathcal{P}(\mathcal{Z})$.

**Corollary**: The portfolio return process is **ergodic** — time averages equal ensemble averages. This justifies backtesting.

**Crisis indicator property**: $\Lambda_t > 0$ implies $\mathbb{E}[V(z_{t+\delta})] > V(z_t)$, i.e., the system is in expectation moving further from equilibrium. This is a **leading indicator** — it triggers before the price impact materializes.

## Implementation

In `online/regime_detector.py`, the crisis indicator $\Lambda_t$ is computed in real-time:
1. Get current latent state $z_t = E(x_t)$ from Encoder
2. Evaluate $V(z_t) = z_t^\top P z_t$ (quadratic form, $P$ learned from training data)
3. Estimate $\mathcal{L}V$ from rolling window of past $z$ values
4. Compute $\Lambda_t = \mathcal{L}V / V$
5. If $\Lambda_t > \lambda_{\text{threshold}}$: trigger risk-off mode in Controller C

In [ ]:
# Demonstrate early warning: Λ_t leads price moves
# Check average Λ in the 20 days BEFORE each crisis
print('Lead time analysis: Λ_t as early warning indicator')
print('-' * 50)

for t_start, t_end, label in crises:
    pre_crisis_start = max(0, t_start - 20)
    pre_crisis_Lambda = Lambda[pre_crisis_start:t_start]
    during_crisis_Lambda = Lambda[t_start:min(t_end, len(Lambda))]
    normal_Lambda = Lambda[max(0, t_start-100):pre_crisis_start]
    
    print(f'\n{label}:')
    print(f'  Normal period Λ:     {np.mean(normal_Lambda):.3f} ± {np.std(normal_Lambda):.3f}')
    print(f'  Pre-crisis (20d) Λ:  {np.mean(pre_crisis_Lambda):.3f} ± {np.std(pre_crisis_Lambda):.3f}')
    print(f'  During crisis Λ:     {np.mean(during_crisis_Lambda):.3f} ± {np.std(during_crisis_Lambda):.3f}')
    signal_ratio = np.mean(pre_crisis_Lambda) / np.mean(normal_Lambda) if np.mean(normal_Lambda) != 0 else float('inf')
    print(f'  Pre-crisis / Normal: {signal_ratio:.2f}x elevation')